In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [6]:
# resizing for efficientnet generally uses (224, 224)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    # it shouldn't matter which side of the frame the action is on
    transforms.RandomHorizontalFlip(),
    # converts the image to a pytorch tensor
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [7]:
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, len(files))

In [8]:
dataset_path = "/kaggle/input/datasets/abdulmananraja/real-life-violence-situations/new_violence"

train_dataset_full = ImageFolder(dataset_path, transform=train_transforms)
val_test_dataset_full = ImageFolder(dataset_path, transform=val_test_transforms)

print("Classes:", train_dataset_full.classes)
print("Class mapping:", train_dataset_full.class_to_idx)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/abdulmananraja/real-life-violence-situations/new_violence'

In [ ]:
total_size = len(train_dataset_full)
train_size = int(0.80 * total_size)
val_size = int(0.10 * total_size)
test_size = total_size - train_size - val_size

print("Total:", total_size)
print("Train:", train_size, "Val:", val_size, "Test:", test_size)

Total: 11073
Train: 8858 Val: 1107 Test: 1108


In [ ]:
# Shuffle indices once, then slice — this way train/val/test never overlap,
# and val/test are pulled from the no-augmentation dataset instance.
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(total_size, generator=generator).tolist()

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]

test_indices = indices[train_size + val_size:]

train_dataset = Subset(train_dataset_full, train_indices)

val_dataset = Subset(val_test_dataset_full, val_indices)
test_dataset = Subset(val_test_dataset_full, test_indices)

print("Train:", len(train_dataset))

print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 8858
Validation: 1107
Test: 1108


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


Train batches: 277
Validation batches: 35
Test batches: 35


In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

In [ ]:
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, 2),
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Using:", device)

Using: cuda


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done, loss: {loss.item():.4f}")

Epoch 1 done, loss: 0.0506
Epoch 2 done, loss: 0.0108
Epoch 3 done, loss: 0.0312
Epoch 4 done, loss: 0.0139
Epoch 5 done, loss: 0.0110


In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=train_dataset_full.classes))

Accuracy: 0.9792418772563177
              precision    recall  f1-score   support

non_violence       0.98      0.98      0.98       511
    violence       0.98      0.98      0.98       597

    accuracy                           0.98      1108
   macro avg       0.98      0.98      0.98      1108
weighted avg       0.98      0.98      0.98      1108



In [ ]:
from PIL import Image

img_path = "fight1.jpeg"  # path to your image

img = Image.open(img_path).convert("RGB")
img_tensor = transform(img).unsqueeze(0).to(device)  # add batch dimension

model.eval()
with torch.no_grad():
    output = model(img_tensor)
    pred = output.argmax(dim=1).item()

print("Predicted class:", train_dataset_full.classes[pred])

FileNotFoundError: [Errno 2] No such file or directory: 'fight1.jpeg'